# Inference - Water Body Segmentation Predictions

This notebook loads a trained semantic segmentation model and generates predictions on new satellite images.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns
from PIL import Image
import json
import warnings

warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from config import get_default_config
from models import create_model
from utils import get_device, load_checkpoint

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 2. Configuration and Model Loading

In [ ]:
# Configuration
config = get_default_config()
device = get_device(config.device.device)

# Define paths
project_root = Path.cwd().parent
data_dir = project_root / 'data'
models_dir = project_root / 'models' / 'checkpoints'
results_dir = project_root / 'results' / 'predictions'
results_dir.mkdir(parents=True, exist_ok=True)

# List available checkpoints
if models_dir.exists():
    checkpoints = list(models_dir.glob('*.pt'))
    print(f"\nAvailable Checkpoints:")
    for i, ckpt in enumerate(checkpoints, 1):
        print(f"  {i}. {ckpt.name}")
else:
    print("No checkpoints directory found. Please train a model first.")
    checkpoints = []

In [ ]:
# Load model
model = create_model(
    model_type=config.model.model_type,
    in_channels=3,
    num_classes=1,
    **config.model.__dict__
)
model = model.to(device)

# Load checkpoint (use the first one or latest)
if checkpoints:
    checkpoint_path = checkpoints[-1]  # Latest checkpoint
    print(f"Loading checkpoint: {checkpoint_path.name}")
    load_checkpoint(model, None, None, checkpoint_path, device)
    model.eval()
    print("Model loaded and set to evaluation mode")
else:
    print("WARNING: No checkpoint found. Model will use random weights.")

## 3. Prediction Function

In [ ]:
def preprocess_image(image_path, img_size=256):
    """Load and preprocess image for inference"""
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Cannot load image: {image_path}")
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size))
    
    # Normalize with ImageNet statistics
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_normalized = (img / 255.0 - mean) / std
    
    img_tensor = torch.from_numpy(img_normalized).float().permute(2, 0, 1).unsqueeze(0)
    return img_tensor, img


def predict(image_path, model, device, threshold=0.5):
    """Generate prediction for a single image"""
    img_tensor, img_original = preprocess_image(image_path)
    img_tensor = img_tensor.to(device)
    
    with torch.no_grad():
        output = model(img_tensor)
        prediction = torch.sigmoid(output)
        prediction = prediction.cpu().numpy()[0, 0]
    
    # Create binary mask
    mask = (prediction > threshold).astype(np.uint8)
    
    return prediction, mask, img_original


def create_overlay(image, mask, alpha=0.3):
    """Create overlay visualization"""
    overlay = image.copy()
    overlay[mask == 1] = [0, 100, 255]  # Red color for water
    result = cv2.addWeighted(image, 1 - alpha, overlay, alpha, 0)
    return result

print("Prediction functions defined successfully.")

## 4. Inference on Test Images

In [ ]:
# Load test images
test_images_dir = data_dir / 'valid_images'
test_images = sorted(test_images_dir.glob('*.jpg'))[:10]  # First 10 images

print(f"Found {len(test_images)} test images")
print(f"Processing first {len(test_images)} images...")

# Perform inference
predictions_data = []

for image_path in tqdm(test_images):
    try:
        pred_prob, pred_mask, img_orig = predict(image_path, model, device, threshold=0.5)
        
        predictions_data.append({
            'image_path': str(image_path),
            'image_name': image_path.name,
            'prediction_prob': pred_prob,
            'prediction_mask': pred_mask,
            'image_original': img_orig,
            'water_percentage': float((pred_mask.sum() / pred_mask.size) * 100)
        })
    except Exception as e:
        print(f"Error processing {image_path.name}: {str(e)}")

print(f"\nSuccessfully processed {len(predictions_data)} images")

## 5. Visualization - Prediction Results

In [ ]:
# Visualize predictions (grid layout)
n_images = min(6, len(predictions_data))
fig, axes = plt.subplots(n_images, 4, figsize=(16, 4*n_images))

if n_images == 1:
    axes = axes.reshape(1, -1)

for idx, pred_data in enumerate(predictions_data[:n_images]):
    img = pred_data['image_original']
    prob = pred_data['prediction_prob']
    mask = pred_data['prediction_mask']
    
    # Original image
    axes[idx, 0].imshow(img)
    axes[idx, 0].set_title(f"Original\n{pred_data['image_name'][:20]}", fontsize=10)
    axes[idx, 0].axis('off')
    
    # Probability map
    im1 = axes[idx, 1].imshow(prob, cmap='hot')
    axes[idx, 1].set_title(f"Probability Map\nWater: {pred_data['water_percentage']:.1f}%", fontsize=10)
    axes[idx, 1].axis('off')
    plt.colorbar(im1, ax=axes[idx, 1], fraction=0.046, pad=0.04)
    
    # Binary mask
    axes[idx, 2].imshow(mask, cmap='RdYlGn_r')
    axes[idx, 2].set_title("Binary Mask (threshold=0.5)", fontsize=10)
    axes[idx, 2].axis('off')
    
    # Overlay
    overlay = create_overlay(img, mask, alpha=0.4)
    axes[idx, 3].imshow(overlay)
    axes[idx, 3].set_title("Overlay Visualization", fontsize=10)
    axes[idx, 3].axis('off')

plt.tight_layout()
plt.savefig(results_dir / 'predictions_grid.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Grid visualization saved to {results_dir / 'predictions_grid.png'}")

## 6. Single Image Detailed Analysis

In [ ]:
# Select first image for detailed analysis
selected_idx = 0
selected = predictions_data[selected_idx]

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# Original image
ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(selected['image_original'])
ax1.set_title('Original Image', fontsize=12, fontweight='bold')
ax1.axis('off')

# Probability heatmap
ax2 = fig.add_subplot(gs[0, 1])
im = ax2.imshow(selected['prediction_prob'], cmap='hot')
ax2.set_title('Prediction Probability Map', fontsize=12, fontweight='bold')
ax2.axis('off')
cbar = plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
cbar.set_label('Probability', rotation=270, labelpad=15)

# Binary mask
ax3 = fig.add_subplot(gs[0, 2])
im2 = ax3.imshow(selected['prediction_mask'], cmap='RdYlGn_r')
ax3.set_title('Binary Mask (threshold=0.5)', fontsize=12, fontweight='bold')
ax3.axis('off')

# Overlay
ax4 = fig.add_subplot(gs[1, 0:2])
overlay = create_overlay(selected['image_original'], selected['prediction_mask'], alpha=0.35)
ax4.imshow(overlay)
ax4.set_title('Overlay Visualization (Water in Red)', fontsize=12, fontweight='bold')
ax4.axis('off')

# Statistics
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
stats_text = f"""
Image: {selected['image_name']}

Statistics:
- Image Size: 256x256 pixels
- Total Pixels: 65,536
- Water Pixels: {selected['prediction_mask'].sum():,}
- Water Coverage: {selected['water_percentage']:.2f}%
- Confidence Range: {selected['prediction_prob'].min():.3f} - {selected['prediction_prob'].max():.3f}
- Mean Probability: {selected['prediction_prob'].mean():.3f}
"""
ax5.text(0.05, 0.95, stats_text, transform=ax5.transAxes, 
         verticalalignment='top', fontfamily='monospace', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.savefig(results_dir / f'detailed_analysis_{selected["image_name"]}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Detailed analysis saved")

## 7. Multi-Threshold Analysis

In [ ]:
# Analyze predictions at different thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
selected_idx = 0
selected_prob = predictions_data[selected_idx]['prediction_prob']

fig, axes = plt.subplots(1, len(thresholds), figsize=(16, 4))

for idx, threshold in enumerate(thresholds):
    mask = (selected_prob > threshold).astype(np.uint8)
    water_pct = (mask.sum() / mask.size) * 100
    
    axes[idx].imshow(mask, cmap='RdYlGn_r')
    axes[idx].set_title(f'Threshold: {threshold}\nWater: {water_pct:.1f}%', fontsize=11, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(results_dir / 'multi_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Multi-threshold analysis completed")

## 8. Water Coverage Distribution

In [ ]:
# Calculate coverage statistics
water_coverages = [p['water_percentage'] for p in predictions_data]
image_names = [p['image_name'][:15] for p in predictions_data]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(range(len(water_coverages)), water_coverages, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Image Index', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Water Coverage (%)', fontsize=11, fontweight='bold')
axes[0].set_title('Water Coverage Distribution Across Images', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 100])

# Add value labels on bars
for i, v in enumerate(water_coverages):
    axes[0].text(i, v + 2, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

# Distribution histogram
axes[1].hist(water_coverages, bins=10, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Water Coverage (%)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1].set_title('Distribution of Water Coverage', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Add statistics
stats_text = f"Mean: {np.mean(water_coverages):.1f}%\nStd: {np.std(water_coverages):.1f}%\nMin: {np.min(water_coverages):.1f}%\nMax: {np.max(water_coverages):.1f}%"
axes[1].text(0.95, 0.95, stats_text, transform=axes[1].transAxes,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
            fontsize=10, fontfamily='monospace')

plt.tight_layout()
plt.savefig(results_dir / 'coverage_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nWater Coverage Statistics:")
print(f"  Mean: {np.mean(water_coverages):.2f}%")
print(f"  Std Dev: {np.std(water_coverages):.2f}%")
print(f"  Min: {np.min(water_coverages):.2f}%")
print(f"  Max: {np.max(water_coverages):.2f}%")

## 9. Save Predictions

In [ ]:
# Save predictions and overlays
print(f"Saving predictions to {results_dir}...")

for idx, pred_data in enumerate(predictions_data):
    img_name = Path(pred_data['image_name']).stem
    
    # Save probability map
    prob_uint8 = (pred_data['prediction_prob'] * 255).astype(np.uint8)
    cv2.imwrite(str(results_dir / f'{img_name}_probability.png'), prob_uint8)
    
    # Save binary mask
    mask_uint8 = (pred_data['prediction_mask'] * 255).astype(np.uint8)
    cv2.imwrite(str(results_dir / f'{img_name}_mask.png'), mask_uint8)
    
    # Save overlay
    overlay = create_overlay(pred_data['image_original'], pred_data['prediction_mask'], alpha=0.35)
    overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(results_dir / f'{img_name}_overlay.png'), overlay_bgr)

print(f"Saved {len(predictions_data)} predictions")
print(f"Output directory: {results_dir}")

## 10. Export Summary Report

In [ ]:
# Create summary report
summary = {
    'model_type': config.model.model_type,
    'inference_device': str(device),
    'total_images_processed': len(predictions_data),
    'threshold': 0.5,
    'water_coverage_statistics': {
        'mean_percentage': float(np.mean(water_coverages)),
        'std_percentage': float(np.std(water_coverages)),
        'min_percentage': float(np.min(water_coverages)),
        'max_percentage': float(np.max(water_coverages))
    },
    'image_statistics': [
        {
            'image_name': p['image_name'],
            'water_percentage': p['water_percentage'],
            'water_pixels': int(p['prediction_mask'].sum()),
            'mean_confidence': float(p['prediction_prob'].mean()),
            'max_confidence': float(p['prediction_prob'].max()),
            'min_confidence': float(p['prediction_prob'].min())
        }
        for p in predictions_data
    ]
}

# Save report
report_path = results_dir / 'inference_summary.json'
with open(report_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Summary report saved to {report_path}")
print(f"\nInference completed successfully!")
print(f"Generated outputs:")
print(f"  - Prediction grids")
print(f"  - Detailed analyses")
print(f"  - Multi-threshold comparisons")
print(f"  - Coverage distributions")
print(f"  - Probability maps")
print(f"  - Binary masks")
print(f"  - Overlay visualizations")
print(f"  - Summary report")